# Bangla Hallucination Detection — v10 "Judge + RAG"

**Pipeline:** two-regime routing → LLM judge (context rows = single-token Yes/No
logprob; closed-book rows = **chain-of-thought + self-consistency, k=5**) →
**RAG** over an attached Bengali-Wikipedia corpus for closed-book rows →
optional low-weight NLI/e5/hand blend → **one regularized threshold per regime**
calibrated on the labelled set → `submission.csv`. No pseudo-labeling.

**Why:** ~46% of the test set is `[NULL]` context (closed-book). Consistency-only
models (BanglaBERT/NLI/e5) are near-random there and cap out ~0.65. World
knowledge comes from the local open LLM (APIs are banned) plus retrieval.

### Attach on Kaggle (Add Data)
1. **Competition data** — labelled samples (`dataset samples.json`/`train.csv`) + `test set.csv`.
2. **Synthetic dev (optional)** — `synthetic_train_5000.csv` (this repo's `synthetic-data/`).
3. **Bengali Wikipedia (for RAG)** — any public parquet/csv/json with a Bengali text
   column (e.g. a `wikimedia/wikipedia` bn export, or a Bangla passages corpus).
   RAG **auto-skips** if none is attached, so the notebook still runs.

Internet must be OFF at submission (re-run policy). All models are open-weight;
declare model + license + this synthetic data per the rulebook.


In [ ]:
# ============================ SETUP ============================
# vLLM pins torch/transformers -> install it FIRST, everything else without -U.
!pip install -q vllm
!pip install -q sentence-transformers faiss-cpu peft accelerate bitsandbytes
!pip install -q git+https://github.com/csebuetnlp/normalizer


In [ ]:
# ============================ CONFIG & TOGGLES ============================
import os, re, gc, json, math, time, glob, hashlib, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    HAS_CUDA = torch.cuda.is_available()
    N_GPU = max(torch.cuda.device_count(), 1) if HAS_CUDA else 0
except Exception:
    torch = None; HAS_CUDA = False; N_GPU = 0

# ---- feature toggles (safe defaults; each stage self-skips if unavailable) ----
ENABLE_JUDGE          = True     # the LLM judge — the core of v10
ENABLE_SELFCONSISTENCY = True    # CoT + majority vote on closed-book rows
SC_SAMPLES            = 5        # samples per closed-book row for self-consistency
ENABLE_RAG            = True     # retrieval over an attached Bengali-Wikipedia dataset
ENABLE_AUX            = False    # NLI/emb/hand features as a low-weight tie-break
W_JUDGE               = 1.0      # judge weight in the final blend (1.0 = judge-only)
METRIC                = 'macro'  # 'macro' | 'hallucinated' (set to what Kaggle shows)
USE_LORA_ADAPTER      = False    # load a QLoRA verifier adapter (see train_lora_v10)
LORA_PATH             = None     # e.g. '/kaggle/input/hallu-lora-v10/adapter'
ENABLE_SYN_EVAL       = False    # also read F1 on the 5000 synthetic dev set (slow)

# ---- judge model ladder: (model, tensor_parallel, max_len, quant) ----
# tried top-to-bottom until one initialises. All are valid HF ids that fit Kaggle
# GPUs. Swap in google/gemma-2-27b-it or CohereForAI/aya-expanse-32b to compare.
JUDGE_LADDER = [
    ('Qwen/Qwen2.5-32B-Instruct-AWQ', 2, 2560, 'awq'),
    ('Qwen/Qwen2.5-14B-Instruct-AWQ', 2, 3072, 'awq'),
    ('Qwen/Qwen2.5-14B-Instruct-AWQ', 1, 2560, 'awq'),
    ('Qwen/Qwen2.5-7B-Instruct-AWQ',  1, 3072, 'awq'),
]
JUDGE_FALLBACK_BNB = 'Qwen/Qwen2.5-14B-Instruct'   # transformers 4-bit if vLLM breaks

EMB_MODEL = 'intfloat/multilingual-e5-base'
NLI_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'

CTX_CLIP, PROMPT_CLIP, RESP_CLIP = 1600, 400, 700
JUDGE_CHUNK = 256                 # rows per vLLM call
RAG_TOPK = 3
RAG_MAX_PASSAGES = 300_000        # cap for memory safety on Kaggle
RAG_PASSAGE_CHARS = 550

def find_file(*names):
    for nm in names:
        for root in ['/kaggle/input', '.', '..', '/kaggle/working']:
            hits = [h for h in glob.glob(os.path.join(root, '**', '*' + nm + '*'), recursive=True)
                    if os.path.isfile(h)]
            if hits:
                return sorted(hits, key=len)[0]
    return None

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(WORK, exist_ok=True)
print('CUDA:', HAS_CUDA, '| GPUs:', N_GPU, '| work:', WORK)


## Core logic (routing, verdict parsing, self-consistency, metrics, calibration)
Unit-tested on CPU — see `test_core.py` in the repo (25/25 passed).

In [ ]:
# -*- coding: utf-8 -*-
"""
v10 core logic — pure python, no GPU / no heavy deps.
Everything here is unit-tested on CPU so the non-LLM parts of the pipeline are
guaranteed correct. The inference notebook embeds these exact functions.
"""
import re, math

# ---------------------------------------------------------------- #
#  text / routing
# ---------------------------------------------------------------- #
NO_CTX = {'', 'nan', 'naN', 'NaN', '[null]', '[NULL]', 'null', 'NULL', 'none', 'None'}

def has_context(ctx):
    if ctx is None:
        return False
    return str(ctx).strip().lower() not in {s.lower() for s in NO_CTX}

def clean_text(t, normalize=None):
    if t is None:
        return ''
    s = str(t).strip()
    if s.lower() in {x.lower() for x in NO_CTX}:
        return ''
    if normalize is not None:
        try:
            return normalize(s)
        except Exception:
            return s
    return s

# ---------------------------------------------------------------- #
#  verdict parsing for CoT + self-consistency
# ---------------------------------------------------------------- #
YES_WORDS = ['yes', 'faithful', 'correct', 'supported', 'true',
             'হ্যাঁ', 'হাঁ', 'সঠিক', 'সত্য', 'সমর্থিত']
NO_WORDS = ['no', 'hallucinat', 'incorrect', 'unsupported', 'false', 'wrong', 'fabricat',
            'না', 'ভুল', 'মিথ্যা', 'অসমর্থিত', 'ভিত্তিহীন']

_VERDICT_RE = re.compile(
    r'(?:verdict|final answer|answer|উত্তর|রায়|সিদ্ধান্ত)\s*[:：\-]?\s*'
    r'(yes|no|হ্যাঁ|না|সঠিক|ভুল|correct|incorrect|faithful|hallucinated)',
    re.IGNORECASE)

def parse_verdict(text):
    """Return 1 (faithful/Yes), 0 (hallucinated/No), or None if unclear."""
    if not text:
        return None
    t = text.strip().lower()
    # 1) explicit "Verdict: X" near the end wins
    matches = _VERDICT_RE.findall(text)
    if matches:
        tok = matches[-1].lower()
        if tok in ('yes', 'সঠিক', 'correct', 'faithful', 'হ্যাঁ'):
            return 1
        if tok in ('no', 'ভুল', 'incorrect', 'hallucinated', 'না'):
            return 0
    # 2) last standalone yes/no token
    last = None
    for m in re.finditer(r'\b(yes|no)\b|(হ্যাঁ|না|সঠিক|ভুল)', t):
        last = m.group(0)
    if last is not None:
        if last in ('yes', 'হ্যাঁ', 'সঠিক'):
            return 1
        if last in ('no', 'না', 'ভুল'):
            return 0
    # 3) fall back to keyword tally
    y = sum(t.count(w) for w in YES_WORDS)
    n = sum(t.count(w) for w in NO_WORDS)
    if y > n:
        return 1
    if n > y:
        return 0
    return None

def self_consistency_pfaithful(samples, fallback=0.5):
    """samples: list of generated strings. Return P(faithful) by majority vote,
    ignoring unparseable samples; fallback if none parse."""
    votes = [parse_verdict(s) for s in samples]
    votes = [v for v in votes if v is not None]
    if not votes:
        return fallback
    return sum(votes) / len(votes)

# ---------------------------------------------------------------- #
#  metrics (pure python; matches sklearn f1_score average='macro' / binary)
# ---------------------------------------------------------------- #
def _f1(y, p, pos):
    tp = fp = fn = 0
    for yi, pi in zip(y, p):
        if pi == pos and yi == pos:
            tp += 1
        elif pi == pos and yi != pos:
            fp += 1
        elif pi != pos and yi == pos:
            fn += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

def macro_f1(y, p):
    return 0.5 * (_f1(y, p, 0) + _f1(y, p, 1))

def score_metric(y, p, metric='macro'):
    if metric == 'macro':
        return macro_f1(y, p)
    if metric == 'hallucinated':      # F1 of the hallucinated class (label 0)
        return _f1(y, p, 0)
    if metric == 'faithful':
        return _f1(y, p, 1)
    raise ValueError(metric)

# ---------------------------------------------------------------- #
#  regularized threshold calibration
# ---------------------------------------------------------------- #
def calibrate_threshold(scores, labels, metric='macro',
                        grid=None, prefer=0.5):
    """Single threshold. Coarse grid, tie-break toward `prefer` (=0.5) to avoid
    overfitting a tiny dev set. pred = 1 if score >= t."""
    if grid is None:
        grid = [round(0.20 + 0.025 * i, 3) for i in range(25)]   # 0.20..0.80
    best_t, best_s = prefer, -1.0
    for t in grid:
        pred = [1 if s >= t else 0 for s in scores]
        sc = score_metric(labels, pred, metric)
        if sc > best_s + 1e-9 or (abs(sc - best_s) <= 1e-9 and abs(t - prefer) < abs(best_t - prefer)):
            best_t, best_s = t, sc
    return best_t, best_s

def calibrate_two_regime(scores, labels, is_ctx, metric='macro'):
    """One threshold for context rows, one for closed-book rows — each calibrated
    only on its own subset, regularized toward 0.5."""
    sc_c = [s for s, g in zip(scores, is_ctx) if g]
    lb_c = [l for l, g in zip(labels, is_ctx) if g]
    sc_n = [s for s, g in zip(scores, is_ctx) if not g]
    lb_n = [l for l, g in zip(labels, is_ctx) if not g]
    t_ctx = calibrate_threshold(sc_c, lb_c, metric)[0] if sc_c else 0.5
    t_null = calibrate_threshold(sc_n, lb_n, metric)[0] if sc_n else 0.5
    return t_ctx, t_null

def route_predict(scores, is_ctx, t_ctx, t_null):
    return [1 if (s >= (t_ctx if g else t_null)) else 0
            for s, g in zip(scores, is_ctx)]

# ---------------------------------------------------------------- #
#  blend (judge-dominant)
# ---------------------------------------------------------------- #
def blend(judge, aux=None, w_judge=0.8):
    """Judge-dominant blend. aux is an optional secondary probability list."""
    if aux is None:
        return list(judge)
    return [w_judge * j + (1 - w_judge) * a for j, a in zip(judge, aux)]


## 1. Load & normalize · route into context vs closed-book

In [ ]:
# ============================ LOAD & NORMALIZE ============================
try:
    from normalizer import normalize
except Exception:
    print('normalizer unavailable -> identity'); normalize = lambda s: s

DATA_PATH = find_file('dataset samples.json', 'dataset_samples.json', 'train.csv')
TEST_PATH = find_file('test set.csv', 'test_set.csv', 'test.csv')
assert DATA_PATH and TEST_PATH, 'attach the competition data (labelled samples + test set)'
print('labelled:', DATA_PATH, '\ntest    :', TEST_PATH)

if DATA_PATH.endswith('.json'):
    df = pd.DataFrame(json.load(open(DATA_PATH, encoding='utf-8')))
else:
    df = pd.read_csv(DATA_PATH)
test_df = pd.read_csv(TEST_PATH)
if 'id' not in test_df.columns:
    test_df.insert(0, 'id', np.arange(1, len(test_df) + 1))

for d in (df, test_df):
    for c in ['context', 'prompt_bn', 'response_bn']:
        d[c] = d[c].apply(lambda x: clean_text(x, normalize))
    d['has_ctx'] = d['context'].apply(has_context)
    d['premise'] = [c if h else q for c, q, h in zip(d['context'], d['prompt_bn'], d['has_ctx'])]

df = df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
y = df['label'].astype(int).tolist()
print(f'labelled n={len(df)}  (faithful={sum(y)}, halluc={len(y)-sum(y)}, with-ctx={int(df.has_ctx.sum())})')
print(f'test     n={len(test_df)}  (with-ctx={int(test_df.has_ctx.sum())}, '
      f'closed-book={int((~test_df.has_ctx).sum())})')

# optional synthetic dev set (for extra validation / few-shot enrichment)
SYN_PATH = find_file('synthetic_train_5000.csv')
syn_df = None
if SYN_PATH:
    syn_df = pd.read_csv(SYN_PATH)
    for c in ['context', 'prompt_bn', 'response_bn']:
        syn_df[c] = syn_df[c].apply(lambda x: clean_text(x, normalize))
    syn_df['has_ctx'] = syn_df['context'].apply(has_context)
    syn_df['premise'] = [c if h else q for c, q, h in
                         zip(syn_df['context'], syn_df['prompt_bn'], syn_df['has_ctx'])]
    print('synthetic dev loaded:', len(syn_df), 'rows')


## 2. Prompts — logprob (context) and CoT (closed-book / RAG)

In [ ]:
# ============================ PROMPT DESIGN ============================
def _clip(s, n):
    s = s or ''
    return s if len(s) <= n else s[:n] + '…'

# few-shot exemplars: shortest 2-per-class closed-book rows from the labelled set
_nc = df[~df.has_ctx].copy()
FEWSHOT_IDX = []
if len(_nc) >= 4:
    _nc['tl'] = _nc.prompt_bn.str.len() + _nc.response_bn.str.len()
    FEWSHOT_IDX = (list(_nc[_nc.label == 1].sort_values('tl').index[:2]) +
                   list(_nc[_nc.label == 0].sort_values('tl').index[:2]))
print('few-shot exemplar rows (excluded from calibration):', FEWSHOT_IDX)

def fewshot_block():
    lines = []
    for i in FEWSHOT_IDX:
        r = df.loc[i]
        v = 'Yes' if int(r.label) == 1 else 'No'
        lines.append(f'Question: {_clip(r.prompt_bn, 200)}\n'
                     f'Proposed answer: {_clip(r.response_bn, 200)}\nVerdict: {v}')
    return '\n\n'.join(lines)

FEWSHOT = fewshot_block()
SYS = ('You are a meticulous bilingual (Bengali/English) fact-checker. '
       'You judge whether a Bengali answer is factually correct and, when a '
       'passage is given, fully supported by it.')

def prompt_logprob(ctx, q, resp):
    """Single-token Yes/No prompt — used for CONTEXT rows (fast, NLI-like)."""
    ctx, q, resp = _clip(ctx, CTX_CLIP), _clip(q, PROMPT_CLIP), _clip(resp, RESP_CLIP)
    if has_context(ctx):
        return (f'Passage (Bengali):\n{ctx}\n\nQuestion (Bengali): {q}\n'
                f'Response (Bengali): {resp}\n\n'
                'Is the response factually correct AND fully supported by the passage, '
                'with no fabricated or contradicting detail? '
                'Answer with exactly one word: Yes or No.')
    head = ('Here are solved examples:\n\n' + FEWSHOT + '\n\n') if FEWSHOT else ''
    return (head + f'Now judge this one.\nQuestion (Bengali): {q}\n'
            f'Proposed answer (Bengali): {resp}\n'
            'Is the proposed answer factually correct? '
            'Answer with exactly one word: Yes or No.\nVerdict:')

def prompt_cot(ctx, q, resp):
    """Reason-then-verdict prompt — used for CLOSED-BOOK (and RAG) rows."""
    ctx, q, resp = _clip(ctx, CTX_CLIP), _clip(q, PROMPT_CLIP), _clip(resp, RESP_CLIP)
    if has_context(ctx):
        return (f'Passage (Bengali):\n{ctx}\n\nQuestion (Bengali): {q}\n'
                f'Response (Bengali): {resp}\n\n'
                'Reason in 1-3 short sentences about whether the response is factually '
                'correct and supported by the passage. Then, on a new line, output exactly '
                '"Verdict: Yes" if it is faithful, or "Verdict: No" if it is hallucinated.')
    head = ('Solved examples:\n\n' + FEWSHOT + '\n\n') if FEWSHOT else ''
    return (head + 'Now judge the following using your own knowledge of Bengali/Bangladesh '
            'facts. Reason in 1-3 short sentences, then on a new line output exactly '
            '"Verdict: Yes" if the answer is factually correct, or "Verdict: No" if it is '
            f'wrong or fabricated.\nQuestion (Bengali): {q}\n'
            f'Proposed answer (Bengali): {resp}')


## 3. Judge engine — vLLM ladder (Qwen2.5-32B→14B→7B AWG) with bnb 4-bit fallback

In [ ]:
# ============================ JUDGE ENGINE (vLLM ladder + bnb fallback) ============================
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

llm = judge_tok = None
JUDGE_BACKEND = JUDGE_NAME = None
JUDGE_MAXLEN = 2560
SamplingParams = None

if ENABLE_JUDGE:
    try:
        from vllm import LLM, SamplingParams
        from transformers import AutoTokenizer
        for name, tp, ml, quant in JUDGE_LADDER:
            if tp > max(N_GPU, 1):
                continue
            try:
                print(f'>>> trying {name} (tp={tp}, max_len={ml}, quant={quant})')
                kw = dict(model=name, tensor_parallel_size=tp, dtype='half',
                          max_model_len=ml, gpu_memory_utilization=0.92,
                          enforce_eager=True, swap_space=2, disable_log_stats=True)
                if quant:
                    kw['quantization'] = quant
                llm = LLM(**kw)
                judge_tok = AutoTokenizer.from_pretrained(name)
                if USE_LORA_ADAPTER and LORA_PATH:
                    print('    (LoRA adapter path set — enable vLLM LoRA or merge weights offline)')
                JUDGE_BACKEND, JUDGE_NAME, JUDGE_MAXLEN = 'vllm', name, ml
                print(f'>>> judge online (vLLM): {name}')
                break
            except Exception as e:
                print(f'    failed: {type(e).__name__}: {str(e)[:200]}')
                llm = None; gc.collect()
                if torch is not None and HAS_CUDA:
                    torch.cuda.empty_cache()
    except Exception as e:
        print('vLLM import failed:', str(e)[:200])

    if JUDGE_BACKEND is None:
        print('>>> vLLM unavailable — falling back to transformers + bitsandbytes 4-bit')
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
            judge_tok = AutoTokenizer.from_pretrained(JUDGE_FALLBACK_BNB)
            if judge_tok.pad_token is None:
                judge_tok.pad_token = judge_tok.eos_token
            bnb_model = AutoModelForCausalLM.from_pretrained(
                JUDGE_FALLBACK_BNB, device_map='auto',
                quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
            bnb_model.eval()
            JUDGE_BACKEND, JUDGE_NAME, JUDGE_MAXLEN = 'bnb', JUDGE_FALLBACK_BNB, 2560
            print('>>> judge online (bnb):', JUDGE_FALLBACK_BNB)
        except Exception as e:
            print('bnb fallback failed:', str(e)[:200], '\n>>> judge DISABLED — will use 0.5 priors')
            ENABLE_JUDGE = False


## 4. Run the judge — logprob on context rows, CoT+self-consistency on closed-book rows

In [ ]:
# ============================ JUDGE SCORING (logprob + CoT/self-consistency) ============================
YES_SET = {'yes', 'y', 'true'}
NO_SET  = {'no', 'n', 'false'}

def _to_chat(user_msg):
    msgs = [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user_msg}]
    return judge_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _fit(txt):
    ids = judge_tok.encode(txt)
    if len(ids) > JUDGE_MAXLEN - 8:
        txt = judge_tok.decode(ids[:JUDGE_MAXLEN - 8])
    return txt

def logprob_pfaithful(user_msgs):
    """P(faithful) from the first generated token's Yes/No logprobs."""
    prompts = [_fit(_to_chat(u)) for u in user_msgs]
    if JUDGE_BACKEND == 'vllm':
        sp = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
        outs = llm.generate(prompts, sp)
        res = []
        for o in outs:
            py = pn = 0.0
            lp = o.outputs[0].logprobs
            if lp:
                for tid, l in lp[0].items():
                    s = (l.decoded_token if l.decoded_token is not None
                         else judge_tok.decode([tid])).strip().lower()
                    if s in YES_SET: py += math.exp(l.logprob)
                    elif s in NO_SET: pn += math.exp(l.logprob)
            res.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
        return res
    # bnb fallback: read last-position logits
    res = []; B = 8
    yes_ids = set(sum([judge_tok.encode(w, add_special_tokens=False)[:1] for w in ['Yes', 'yes', ' Yes']], []))
    no_ids  = set(sum([judge_tok.encode(w, add_special_tokens=False)[:1] for w in ['No', 'no', ' No']], []))
    for s0 in range(0, len(prompts), B):
        enc = judge_tok(prompts[s0:s0 + B], return_tensors='pt', padding=True,
                        truncation=True, max_length=JUDGE_MAXLEN).to(bnb_model.device)
        with torch.no_grad():
            logits = bnb_model(**enc).logits
        last = enc['attention_mask'].sum(1) - 1
        for bi in range(logits.shape[0]):
            probs = torch.softmax(logits[bi, last[bi]].float(), dim=-1)
            py = float(sum(probs[i] for i in yes_ids)); pn = float(sum(probs[i] for i in no_ids))
            res.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
    return res

def cot_pfaithful(user_msgs, n=SC_SAMPLES):
    """P(faithful) by CoT reasoning + majority vote over n samples."""
    if JUDGE_BACKEND != 'vllm':
        return logprob_pfaithful(user_msgs)   # SC needs sampling; bnb -> logprob path
    prompts = [_fit(_to_chat(u)) for u in user_msgs]
    sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=220, n=n)
    outs = llm.generate(prompts, sp)
    return [self_consistency_pfaithful([c.text for c in o.outputs]) for o in outs]

def _chunked(fn, msgs, **kw):
    out = []
    for s0 in range(0, len(msgs), JUDGE_CHUNK):
        out += fn(msgs[s0:s0 + JUDGE_CHUNK], **kw)
        print(f'    {min(s0 + JUDGE_CHUNK, len(msgs))}/{len(msgs)}')
    return out

def judge_frame(frame, tag):
    """Returns dict index -> P(faithful). Context rows use logprob; closed-book
    rows use CoT+self-consistency (or logprob if SC disabled)."""
    t0 = time.time()
    scores = {}
    if not ENABLE_JUDGE:
        return {i: 0.5 for i in frame.index}
    ctx_idx  = [i for i in frame.index if frame.at[i, 'has_ctx']]
    null_idx = [i for i in frame.index if not frame.at[i, 'has_ctx']]

    print(f'[{tag}] context rows: {len(ctx_idx)} (logprob)')
    if ctx_idx:
        msgs = [prompt_logprob(frame.at[i, 'context'], frame.at[i, 'prompt_bn'],
                               frame.at[i, 'response_bn']) for i in ctx_idx]
        for i, s in zip(ctx_idx, _chunked(logprob_pfaithful, msgs)):
            scores[i] = s

    print(f'[{tag}] closed-book rows: {len(null_idx)} '
          f'({"CoT x%d" % SC_SAMPLES if ENABLE_SELFCONSISTENCY else "logprob"})')
    if null_idx:
        if ENABLE_SELFCONSISTENCY:
            msgs = [prompt_cot('', frame.at[i, 'prompt_bn'], frame.at[i, 'response_bn'])
                    for i in null_idx]
            fn = cot_pfaithful
        else:
            msgs = [prompt_logprob('', frame.at[i, 'prompt_bn'], frame.at[i, 'response_bn'])
                    for i in null_idx]
            fn = logprob_pfaithful
        for i, s in zip(null_idx, _chunked(fn, msgs)):
            scores[i] = s

    print(f'[{tag}] judged in {(time.time() - t0) / 60:.1f} min')
    return scores

judge_train = judge_frame(df, 'train')
judge_test  = judge_frame(test_df, 'test')
df['judge'] = [judge_train[i] for i in df.index]
test_df['judge'] = [judge_test[i] for i in test_df.index]

# quick judge-alone sanity on the labelled set, per regime
if ENABLE_JUDGE:
    for g, sub in df.groupby('has_ctx'):
        pr = [1 if s >= 0.5 else 0 for s in sub['judge']]
        print(f'  has_ctx={g}: judge-alone macroF1={macro_f1(sub["label"].tolist(), pr):.4f} (n={len(sub)})')


## 5. RAG — retrieve Bengali-Wikipedia evidence and re-judge closed-book rows (auto-skips if no corpus)

In [ ]:
# ============================ RAG over Bengali Wikipedia (closed-book rows) ============================
# Attach a public Bengali-Wikipedia dataset on Kaggle (any parquet/csv/json with a
# text column, e.g. 'wikimedia/wikipedia' bn split exported to parquet, or
# 'bengali-wikipedia-passages'). This stage self-skips if none is found or anything
# fails — the judge scores from the previous cell remain the fallback.
RAG_OK = False
RAG_BLEND = 0.5          # final_null = (1-b)*closed_book_judge + b*rag_judge

def _find_wiki():
    cand = []
    for root in ['/kaggle/input']:
        for ext in ('*.parquet', '*.csv', '*.json', '*.jsonl'):
            cand += glob.glob(os.path.join(root, '**', ext), recursive=True)
    # prefer files that look like a wiki / passages corpus, and NOT the competition data
    bad = ('test set', 'test_set', 'dataset samples', 'sample submission', 'synthetic_train')
    cand = [c for c in cand if not any(b in c.lower() for b in bad)]
    cand = [c for c in cand if any(k in c.lower() for k in ('wiki', 'passage', 'corpus', 'bangla', 'bengali'))]
    return sorted(cand, key=lambda p: (-os.path.getsize(p), len(p)))

def _load_passages(path):
    txtcols = ('text', 'passage', 'content', 'body', 'context', 'article', 'paragraph')
    if path.endswith('.parquet'):
        d = pd.read_parquet(path)
    elif path.endswith(('.json', '.jsonl')):
        d = pd.read_json(path, lines=path.endswith('.jsonl'))
    else:
        d = pd.read_csv(path)
    col = next((c for c in d.columns if c.lower() in txtcols), None)
    if col is None:
        col = max(d.columns, key=lambda c: d[c].astype(str).str.len().mean())
    passages = []
    for t in d[col].astype(str).tolist():
        t = clean_text(t, normalize)
        if len(t) < 40:
            continue
        for s0 in range(0, len(t), RAG_PASSAGE_CHARS):
            passages.append(t[s0:s0 + RAG_PASSAGE_CHARS])
            if len(passages) >= RAG_MAX_PASSAGES:
                return passages
    return passages

if ENABLE_RAG and ENABLE_JUDGE and JUDGE_BACKEND == 'vllm':
    try:
        wikis = _find_wiki()
        assert wikis, 'no Bengali-Wikipedia dataset attached'
        print('RAG corpus:', wikis[0])
        passages = _load_passages(wikis[0])
        assert len(passages) > 50, 'corpus too small'
        print(f'RAG passages: {len(passages)}')

        from sentence_transformers import SentenceTransformer
        emb = SentenceTransformer(EMB_MODEL, device='cuda' if HAS_CUDA else 'cpu')
        P = emb.encode(['passage: ' + p for p in passages], batch_size=128,
                       normalize_embeddings=True, show_progress_bar=True).astype('float32')

        try:
            import faiss
            index = faiss.IndexFlatIP(P.shape[1]); index.add(P)
            def _search(qv, k):
                _, I = index.search(qv, k); return I
        except Exception:
            print('faiss unavailable -> numpy top-k')
            def _search(qv, k):
                sims = qv @ P.T
                return np.argpartition(-sims, min(k, sims.shape[1] - 1), axis=1)[:, :k]

        def rag_rejudge(frame, tag):
            idx = [i for i in frame.index if not frame.at[i, 'has_ctx']]
            if not idx:
                return {}
            Q = emb.encode(['query: ' + frame.at[i, 'prompt_bn'] for i in idx],
                           batch_size=128, normalize_embeddings=True,
                           show_progress_bar=False).astype('float32')
            I = _search(Q, RAG_TOPK)
            msgs = []
            for row_n, i in enumerate(idx):
                ev = '\n'.join(passages[j] for j in I[row_n] if 0 <= j < len(passages))
                msgs.append(prompt_cot(ev, frame.at[i, 'prompt_bn'], frame.at[i, 'response_bn']))
            print(f'[RAG {tag}] re-judging {len(idx)} closed-book rows with retrieved evidence')
            sc = _chunked(cot_pfaithful, msgs)
            return dict(zip(idx, sc))

        rag_train = rag_rejudge(df, 'train')
        rag_test  = rag_rejudge(test_df, 'test')
        for frame, rag in [(df, rag_train), (test_df, rag_test)]:
            newv = []
            for i in frame.index:
                if i in rag:
                    newv.append((1 - RAG_BLEND) * frame.at[i, 'judge'] + RAG_BLEND * rag[i])
                else:
                    newv.append(frame.at[i, 'judge'])
            frame['judge'] = newv
        del emb, P; gc.collect()
        if HAS_CUDA: torch.cuda.empty_cache()
        RAG_OK = True
        print('RAG applied to closed-book rows.')
    except Exception as e:
        print('RAG skipped:', type(e).__name__, str(e)[:200])
else:
    print('RAG disabled or judge not on vLLM — skipping.')


## 6. Optional aux features (NLI + e5 + hand + LR) — low-weight tie-break, OFF by default

In [ ]:
# ============================ OPTIONAL AUX FEATURES (low-weight tie-break) ============================
# Default OFF (W_JUDGE=1.0). Turn on ENABLE_AUX and set W_JUDGE<1.0 only if the
# judge-vs-blend comparison in the next cell shows the blend wins on the dev set.
aux_train = [0.5] * len(df)
aux_test  = [0.5] * len(test_df)

if ENABLE_AUX:
    try:
        # free the judge to make room for the small encoders
        if ENABLE_JUDGE and JUDGE_BACKEND == 'vllm':
            try:
                from vllm.distributed import destroy_model_parallel, destroy_distributed_environment
                del llm; destroy_model_parallel(); destroy_distributed_environment()
            except Exception:
                pass
        gc.collect()
        if HAS_CUDA: torch.cuda.empty_cache()

        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        from sentence_transformers import SentenceTransformer
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline as SKPipe
        dev = 'cuda' if HAS_CUDA else 'cpu'

        # NLI entailment / contradiction
        ntok = AutoTokenizer.from_pretrained(NLI_MODEL)
        nmod = AutoModelForSequenceClassification.from_pretrained(
            NLI_MODEL, torch_dtype=torch.float16 if HAS_CUDA else torch.float32).to(dev).eval()
        def nli_feats(fr):
            ent, con = [], []
            for s0 in range(0, len(fr), 32):
                sub = fr.iloc[s0:s0 + 32]
                enc = ntok(list(sub['premise']), list(sub['response_bn']), truncation=True,
                           max_length=512, padding=True, return_tensors='pt').to(dev)
                with torch.no_grad():
                    p = torch.softmax(nmod(**enc).logits.float(), -1).cpu().numpy()
                ent += list(p[:, 0]); con += list(p[:, 2])
            return np.array(ent), np.array(con)
        e_tr, c_tr = nli_feats(df); e_te, c_te = nli_feats(test_df)
        del nmod; gc.collect()
        if HAS_CUDA: torch.cuda.empty_cache()

        # e5 similarity
        em = SentenceTransformer(EMB_MODEL, device=dev)
        def sim(fr):
            r = em.encode(['query: ' + t for t in fr['response_bn']], normalize_embeddings=True)
            p = em.encode(['passage: ' + t for t in fr['premise']], normalize_embeddings=True)
            return np.sum(r * p, axis=1)
        s_tr, s_te = sim(df), sim(test_df)
        del em; gc.collect()
        if HAS_CUDA: torch.cuda.empty_cache()

        def hand(fr):
            out = []
            for _, r in fr.iterrows():
                rt = set(re.findall(r'[ঀ-৿a-zA-Z0-9]+', r['response_bn']))
                pt = set(re.findall(r'[ঀ-৿a-zA-Z0-9]+', r['premise']))
                jac = len(rt & pt) / max(len(rt | pt), 1)
                out.append([jac, len(rt - pt) / max(len(rt), 1), float(len(rt) <= 1), float(r['has_ctx'])])
            return np.array(out)
        h_tr, h_te = hand(df), hand(test_df)

        X_tr = np.column_stack([df['judge'].values, e_tr, c_tr, s_tr, h_tr])
        X_te = np.column_stack([test_df['judge'].values, e_te, c_te, s_te, h_te])
        lr = SKPipe([('sc', StandardScaler()),
                     ('lr', LogisticRegression(C=0.3, max_iter=2000, class_weight='balanced'))])
        lr.fit(X_tr, y)
        aux_train = lr.predict_proba(X_tr)[:, 1].tolist()
        aux_test  = lr.predict_proba(X_te)[:, 1].tolist()
        print('aux features + LR ready')
    except Exception as e:
        print('aux skipped:', type(e).__name__, str(e)[:200])
else:
    print('aux disabled (judge-only).')


## 7. Calibrate per-regime thresholds, compare judge-vs-blend, write submission

In [ ]:
# ============================ CALIBRATE, COMPARE, SUBMIT ============================
# final per-row score = judge-dominant blend (judge only when W_JUDGE=1.0)
final_train = blend(df['judge'].tolist(), aux_train, W_JUDGE)
final_test  = blend(test_df['judge'].tolist(), aux_test,  W_JUDGE)

is_ctx_tr = df['has_ctx'].tolist()
is_ctx_te = test_df['has_ctx'].tolist()

# calibrate on the labelled rows, EXCLUDING few-shot exemplars, with a single
# regularized threshold per regime (context / closed-book)
cal_mask = [i not in set(FEWSHOT_IDX) for i in df.index]
cal_scores = [s for s, m in zip(final_train, cal_mask) if m]
cal_labels = [l for l, m in zip(y, cal_mask) if m]
cal_ctx    = [g for g, m in zip(is_ctx_tr, cal_mask) if m]

t_ctx, t_null = calibrate_two_regime(cal_scores, cal_labels, cal_ctx, METRIC)

# honest readout: judge-only vs blend on the labelled set (same thresholds search)
def _readout(scores, tag):
    tc, tn = calibrate_two_regime([s for s, m in zip(scores, cal_mask) if m],
                                  cal_labels, cal_ctx, METRIC)
    pred = route_predict([s for s, m in zip(scores, cal_mask) if m], cal_ctx, tc, tn)
    print(f'  {tag:11s}: thr(ctx={tc:.3f}, null={tn:.3f})  '
          f'{METRIC}-F1={score_metric(cal_labels, pred, METRIC):.4f}')
print('labelled-set calibration readout (optimistic, tiny set):')
_readout(df['judge'].tolist(), 'judge-only')
if ENABLE_AUX:
    _readout(final_train, 'judge+aux')

# optional: synthetic dev readout (needs the synthetic set judged; off by default)
if ENABLE_SYN_EVAL and syn_df is not None and ENABLE_JUDGE:
    try:
        js = judge_frame(syn_df, 'synthetic')
        sp = route_predict([js[i] for i in syn_df.index], syn_df['has_ctx'].tolist(), t_ctx, t_null)
        print(f'  synthetic dev {METRIC}-F1='
              f'{score_metric(syn_df["label"].astype(int).tolist(), sp, METRIC):.4f} (n={len(syn_df)})')
    except Exception as e:
        print('syn eval skipped:', str(e)[:120])

# ---- predict test & write submission ----
test_pred = route_predict(final_test, is_ctx_te, t_ctx, t_null)
sub = pd.DataFrame({'id': test_df['id'].values, 'label': test_pred})
SUB_PATH = os.path.join(WORK, 'submission.csv')
sub.to_csv(SUB_PATH, index=False)

print(f'\nwrote {SUB_PATH}  ({len(sub)} rows)  '
      f'judge={JUDGE_NAME}  RAG={"on" if RAG_OK else "off"}  '
      f'SC={"on" if ENABLE_SELFCONSISTENCY else "off"}')
print('deployment thresholds: ctx=%.3f  closed-book=%.3f' % (t_ctx, t_null))
vc = sub['label'].value_counts(normalize=True)
print('prediction mix -> faithful(1)=%.3f  hallucinated(0)=%.3f'
      % (vc.get(1, 0.0), vc.get(0, 0.0)))
for g, s in sub.groupby(test_df['has_ctx'].values)['label']:
    print(f'  has_ctx={g}: mean(label)={s.mean():.3f}  n={len(s)}')
print('\nSanity: labelled set is ~55% faithful. If a whole regime collapses to one '
      'class, re-check the judge sanity print and the METRIC flag before submitting.')


## Summary (for the report / 7-min video)

Judge (Qwen2.5-AWQ via vLLM) with **two regimes** — single-token logprob for
grounded rows, **CoT + self-consistency** for closed-book rows — then **RAG**
over Bengali Wikipedia to inject world knowledge into the 46%% closed-book slice,
then **one regularized threshold per regime**. Pseudo-labeling and the heavy
ML stack were dropped (they overfit the tiny labelled set / hurt private LB).
Novelty: regime-aware judging + retrieval-grounded closed-book fact-checking for
low-resource Bengali. Fine-tuning path in `train_lora_v10.ipynb`.
